# 第二十九课:连续批处理——推理服务的吞吐引擎

讲义:[29-continuous-batching.md](29-continuous-batching.md)。推理篇开场:真实测量 prefill/decode 成本结构,手撕 static / continuous / 成组准入三种调度器,三种负载场景实测。

In [ ]:
import math

import time

import numpy as np



# ===== 第 20 课 MiniGPT 原码(含 generate) =====

def softmax(x, axis=-1):

    m = np.max(x, axis=axis, keepdims=True)

    e = np.exp(x - m)

    return e / e.sum(axis=axis, keepdims=True)



class MiniGPT:

    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):

        rng = np.random.default_rng(seed)

        s = 0.1

        self.E = rng.normal(size=(vocab, d)) * 0.5

        self.dh = d // n_heads

        self.H, self.D, self.NL = n_heads, d, n_layers

        self.layers = [{

            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,

            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,

            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,

            'W3': rng.normal(size=(d, 4*d))*s*0.8,

        } for _ in range(n_layers)]

        self.head = rng.normal(size=(d, vocab)) * s



    def rms(self, x, eps=1e-5):

        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)



    def rope(self, x, pos):

        half = x.shape[-1] // 2

        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))

        c, s = np.cos(ang), np.sin(ang)

        x1, x2 = x[..., :half], x[..., half:]

        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)



    def _attn(self, x, pos, W, cache):

        n = x.shape[0]

        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']

        if cache is not None:

            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])

        T = k.shape[0]

        mask = np.triu(np.ones((n, T), bool), T - n + 1)

        outs = []

        for h in range(self.H):

            sl = slice(h*self.dh, (h+1)*self.dh)

            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)

            s = np.where(mask, -np.inf, s)

            outs.append(softmax(s) @ v[:, sl])

        return np.concatenate(outs, -1) @ W['Wo'], (k, v)



    def _ffn(self, x, W):

        h = x @ W['W1']

        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']



    def forward(self, ids, cache=None):

        pos0 = cache[0][0].shape[0] if cache else 0

        x = self.E[list(ids)]

        new_cache = []

        for i, W in enumerate(self.layers):

            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,

                               cache[i] if cache else None)

            x = x + a

            x = x + self._ffn(self.rms(x), W)

            new_cache.append(kv)

        logits = self.rms(x) @ self.head

        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'

        return logits, new_cache



    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):

        rng = rng or np.random.default_rng()

        logits, cache = self.forward(ids)

        out = []

        for _ in range(n):

            p = softmax(logits[-1] / T)

            if top_p < 1.0:

                order = np.argsort(-p); cum = np.cumsum(p[order])

                keep = order[:np.searchsorted(cum, top_p)+1]

                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()

            nxt = int(rng.choice(len(p), p=p/p.sum()))

            out.append(nxt)

            logits, cache = self.forward([nxt], cache)

        return list(ids) + out

In [ ]:
# ===== 实验 1:真实成本结构(讲义 §2) =====
import time

text = open("data/xiyouji.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
V = len(chars)
gpt = MiniGPT(vocab=V, d=64, n_heads=4, n_layers=2, seed=0)

def bench(fn, n=50):
    fn()
    t0 = time.time()
    for _ in range(n):
        fn()
    return (time.time() - t0) / n * 1000   # ms

P = 64
ids64 = list(np.random.default_rng(0).integers(0, V, P))
t_prefill = bench(lambda: gpt.forward(ids64))
ids5 = list(np.random.default_rng(1).integers(0, V, 5))
_, cache = gpt.forward(ids5)
t_decode = bench(lambda: gpt.forward([7], cache))
t_full64 = bench(lambda: gpt.forward(ids64 + [7]))

print(f"[真实成本] MiniGPT d=64 2层,本机 numpy:")
print(f"  prefill 一次性吃 {P} token : {t_prefill:.2f} ms({t_prefill/P*1000:.0f} µs/token,并行)")
print(f"  decode 带 cache 单步      : {t_decode:.2f} ms(只算 1 个 token)")
print(f"  decode 无 cache 全量重算  : {t_full64:.2f} ms(浪费 {t_full64/t_decode:.0f}×)")

In [ ]:
# ===== 模拟器:static / continuous / 成组准入(讲义 §3–4) =====
per_tok = t_prefill / 64

def simulate(mode, n_req, max_batch=8, seed=0, arrival_gap=0.0, group_admit=False):
    """成本模型:prefill ∝ prompt(同批并行取 max);decode 每步近常数(GPU 批内并行)。"""
    rng = np.random.default_rng(seed)
    reqs = [dict(id=i, arrive=i * arrival_gap,
                 prompt=int(rng.integers(16, 65)),
                 remaining=int(rng.integers(8, 41)),
                 ttft=None, done=None) for i in range(n_req)]
    t, done_count, idle, busy = 0.0, 0, 0, 0
    iters_since_admit = 99
    running, waiting = [], list(reqs)
    while done_count < n_req:
        while waiting and waiting[0]['arrive'] <= t:
            running.append(waiting.pop(0))
        if mode == 'static':
            if len(running) < max_batch and waiting:
                t = waiting[0]['arrive']          # 拼车等客
                continue
            batch, running = running[:max_batch], running[max_batch:]
        else:
            batch, running = running[:max_batch], running[max_batch:]
            if not batch:
                t = waiting[0]['arrive']
                continue
        pre_cost = per_tok * max(r['prompt'] for r in batch)
        for r in batch:
            if r['ttft'] is None:
                r['ttft'] = t + pre_cost
        t += pre_cost
        while batch:
            t += t_decode
            for r in batch[:]:
                r['remaining'] -= 1
                if r['remaining'] <= 0:
                    r['done'] = t
                    batch.remove(r)
                    done_count += 1
                    if mode == 'continuous':
                        while waiting and waiting[0]['arrive'] <= t \
                              and len(running) < max_batch:
                            running.append(waiting.pop(0))
            if mode == 'continuous':
                take = max_batch - len(batch)
                iters_since_admit += 1
                if group_admit and 0 < take < 3 and iters_since_admit < 8:
                    take = 0
                admitted = running[:take]
                if admitted:
                    pc = per_tok * max(r['prompt'] for r in admitted)
                    for r in admitted:
                        if r['ttft'] is None:
                            r['ttft'] = t + pc
                    t += pc
                    iters_since_admit = 0
                batch += admitted
                running = running[take:]
            idle += max_batch - len(batch)
            busy += len(batch)
    ttfts = [r['ttft'] - r['arrive'] for r in reqs]
    lats = [r['done'] - r['arrive'] for r in reqs]
    return dict(makespan=t, mean_ttft=np.mean(ttfts), mean_lat=np.mean(lats),
                p99_lat=np.percentile(lats, 99), util=busy / (busy + idle))

print("[闭环] 60 请求全部 t=0 到达(吞吐净对比):")
for name, mode, kw in [('static', 'static', {}), ('continuous', 'continuous', {}),
                       ('continuous成组', 'continuous', dict(group_admit=True))]:
    m = simulate(mode, 60, **kw)
    print(f"  {name:14s}: makespan {m['makespan']:6.1f}ms | 平均时延 {m['mean_lat']:6.1f}"
          f" | p99 {m['p99_lat']:6.1f} | 批占用率 {m['util']:.0%}")

print("\n[轻载] 40 请求,到达间隔 6ms:")
for mode in ('static', 'continuous'):
    m = simulate(mode, 40, arrival_gap=6.0)
    print(f"  {mode:12s}: 平均TTFT {m['mean_ttft']:5.1f} | 平均时延 {m['mean_lat']:6.1f}"
          f" | p99 {m['p99_lat']:6.1f}")

print("\n[重载] 60 请求,到达间隔 1.5ms:")
for mode in ('static', 'continuous'):
    m = simulate(mode, 60, arrival_gap=1.5)
    print(f"  {mode:12s}: 平均TTFT {m['mean_ttft']:5.1f} | 平均时延 {m['mean_lat']:6.1f}"
          f" | p99 {m['p99_lat']:6.1f}")

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 默写 continuous 分支(到达/下车/即补/成组)

# TODO(C2): 加 slo 参数,统计违约率,画批大小 vs 违约率

# TODO(C3): 到达改泊松(指数间隔),重跑轻载

# TODO(E1): max_batch ∈ {2,4,8,16,32} 扫描,画吞吐-时延帕累托前沿
